#**Phase 2 — Frame Extraction, Dataset Structuring, and  Data Loaders

Owner: Abbas




## 1. Project Overview & Objectives
In this phase, we establish a robust, leak-free data pipeline for the **Nexar Collision Prediction**

### Core Objectives:
*   **Environment & Path Verification:** Ensure Google Drive is mounted and check that all required directories exist.
*   **Metadata Integration:** Validate and inspect the structured CSV files containing labels, paths, and training/validation splits.
*   **Video-Level Data Splitting:** Verify that frame splits align strictly with video-level splits to prevent information leakage (Data Leakage) between the training and validation sets.
*   **Custom PyTorch Dataset Construction:** Design a custom `Dataset` class that handles image loading dynamically, converts paths safely, and includes basic exception handling to prevent pipeline crashes.
*   **Data Transformation & Normalization:** Apply standard ImageNet normalization and spatial resizing ($224 \times 224$) suitable for pretrained feature extractors.
*   **Pipeline Dry-Run:** Test the final PyTorch `DataLoaders` to confirm batch shapes, target types, and numerical ranges.


## 2. Environment Verification
We mount Google Drive to access the preprocessed collision dataset and define the project root path.


In [1]:
#Mount Drive and Check Paths
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/AI_Project2/outputs/cv")

print("Drive mounted? ", Path("/content/drive").exists())
print("MyDrive exists? ", Path("/content/drive/MyDrive").exists())
print("ROOT exists? ", ROOT.exists())
print("ROOT path: ", ROOT)


Mounted at /content/drive
Drive mounted?  True
MyDrive exists?  True
ROOT exists?  True
ROOT path:  /content/drive/MyDrive/AI_Project2/outputs/cv


## 3. Metadata and Artifact Verification
We inspect the directory containing our project outputs to ensure all required metadata tables (`dataset_master.csv`, `train_split.csv`, `validation_split.csv`) and configuration parameters (`dataset_mean_std.json`) are present and intact.


In [3]:
# inspect existing arfifacts
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/AI_Project2/outputs/cv")

if ROOT.exists():
    print("CSV files in ROOT:")
    for p in sorted(ROOT.glob("*.csv")):
        print(" -", p.name, "|", p.stat().st_size, "bytes")

    print("\nJSON files in ROOT:")
    for p in sorted(ROOT.glob("*.json")):
        print(" -", p.name, "|", p.stat().st_size, "bytes")

    print("\nLooking for required files:")
    required_files = [
        "dataset_master.csv",
        "train_split.csv",
        "validation_split.csv",
        "dataset_mean_std.json",
    ]

    for fname in required_files:
        fpath = ROOT / fname
        print(f" - {fname}: {fpath.exists()}")
else:
    print("ROOT path does not exist.")


CSV files in ROOT:
 - dataset_master.csv | 847604 bytes
 - df_master.csv | 900734 bytes
 - df_train.csv | 812034 bytes
 - df_val.csv | 88742 bytes
 - processed_frames_log.csv | 2901568 bytes
 - test_split.csv | 436338 bytes
 - train_split.csv | 317 bytes
 - validation_split.csv | 94 bytes
 - video_metadata.csv | 25577 bytes

JSON files in ROOT:
 - dataset_mean_std.json | 131 bytes
 - exploration_summary.json | 1759 bytes

Looking for required files:
 - dataset_master.csv: True
 - train_split.csv: True
 - validation_split.csv: True
 - dataset_mean_std.json: True


## 4. Physical Frame Check
We verify the physical existence of the processed frames directory (`frames_processed`) and count the extracted `.jpg` images to ensure the extraction pipeline completed successfully.


In [4]:
# verify frames directory
from pathlib import Path

FRAMES_DIR = Path("/content/drive/MyDrive/AI_Project2/outputs/cv/frames_processed")

print("FRAMES_DIR exists:", FRAMES_DIR.exists())

jpg_files = list(FRAMES_DIR.rglob("*.jpg")) if FRAMES_DIR.exists() else []
print("Number of .jpg files:", len(jpg_files))

if jpg_files:
    print("Sample image path:", jpg_files[0])


FRAMES_DIR exists: True
Number of .jpg files: 9227
Sample image path: /content/drive/MyDrive/AI_Project2/outputs/cv/frames_processed/test-private/negative/01047/000000.jpg


## 5. Dataset Pipeline Construction

### Preventing Data Leakage
To train a model that generalizes well to unseen videos, we must split our data at the **video level**, not the individual frame level.
*   **Training Set:** Frames belonging strictly to the 80 training videos (`train_split.csv`).
*   **Validation Set:** Frames belonging strictly to the 20 validation videos (`validation_split.csv`).

This guarantees that frames from the same video do not appear in both splits.

### Custom Dataset Design (`NexarFrameDataset`)
*   **Path Resolution:** Safely maps relative frame paths to absolute paths.
*   **Exception Handling:** Employs a `try-except` block during image loading; if a frame is corrupted, it returns a blank placeholder tensor instead of crashing the training process.
*   **Normalization:** Employs custom values loaded from `dataset_mean_std.json`.


In [6]:
#PyTorch Dataset & DataLoaders Definition

import json
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# 1. Define Paths
ROOT = Path("/content/drive/MyDrive/AI_Project2/outputs/cv")
FRAMES_DIR = ROOT / "frames_processed"
DATASET_MASTER = ROOT / "dataset_master.csv"
TRAIN_SPLIT = ROOT / "train_split.csv"
VAL_SPLIT = ROOT / "validation_split.csv"
MEAN_STD_PATH = ROOT / "dataset_mean_std.json"

# 2. Load Normalization Parameters
with open(MEAN_STD_PATH, "r") as f:
    mean_std = json.load(f)
    IMAGENET_MEAN = mean_std["mean"]
    IMAGENET_STD = mean_std["std"]

# 3. Define PyTorch Dataset
class NexarFrameDataset(Dataset):
    def __init__(self, df, frames_dir, transform=None):
        """
        Args:
            df (pd.DataFrame): Dataframe containing 'frame_path' and 'label' columns.
            frames_dir (Path): Base directory where frames are stored.
            transform (callable, optional): PyTorch transforms to apply to images.
        """
        self.df = df.reset_index(drop=True)
        self.frames_dir = frames_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, path_str):
        # Resolve absolute/relative paths safely
        if path_str.startswith("/"):
            return Path(path_str)
        if path_str.startswith("frames_processed/") or path_str.startswith("frames_processed\\"):
            return self.frames_dir.parent / path_str
        return self.frames_dir / path_str

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_path(row["frame_path"])
        label = int(row["label"])

        try:
            # Load image and convert to RGB
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Warning: Failed to load image at {img_path} (Index: {idx}). Error: {e}")
            # Return a dummy tensor if image loading fails, to prevent pipeline crash
            img = Image.new("RGB", (320, 180), color=0)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

# 4. Prepare Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # No augmentation for validation
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 5. Load Splits and Filter Master Dataframe
df_master = pd.read_csv(DATASET_MASTER)
train_videos = pd.read_csv(TRAIN_SPLIT)["video_id"].unique()
val_videos = pd.read_csv(VAL_SPLIT)["video_id"].unique()

# Filter frames based on the video split (No Leakage)
df_train_frames = df_master[df_master["video_id"].isin(train_videos)].copy()
df_val_frames = df_master[df_master["video_id"].isin(val_videos)].copy()

print(f"Total train frames: {len(df_train_frames)} (from {len(train_videos)} videos)")
print(f"Total validation frames: {len(df_val_frames)} (from {len(val_videos)} videos)")

# 6. Instantiate Datasets
train_dataset = NexarFrameDataset(df_train_frames, FRAMES_DIR, transform=train_transforms)
val_dataset = NexarFrameDataset(df_val_frames, FRAMES_DIR, transform=val_transforms)

# 7. Instantiate DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 2  # Colab standard

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\nDataLoaders created successfully!")
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")


Total train frames: 5999 (from 80 videos)
Total validation frames: 1591 (from 20 videos)

DataLoaders created successfully!
Train batches: 188 | Validation batches: 50


## 6. Pipeline Sanity Check & Verification
We execute a dry run of the training data loader to inspect the properties of the generated batches:
*   **Shape:** Expected batch shape is `(Batch Size, Channels, Height, Width)` $\rightarrow [32, 3, 224, 224]$.
*   **Data Types:** `images` must be `torch.float32` and `labels` must be `torch.int64` (required for standard PyTorch classification loss functions).
*   **Value Range:** Since standard ImageNet normalization is applied ($\mu \approx [0.485, 0.456, 0.406]$, $\sigma \approx [0.229, 0.224, 0.225]$), the output pixel values should lie approximately within the $[-2.1, 2.6]$ range, verifying that normalization was successfully executed.


In [7]:
images, labels = next(iter(train_loader))

print("=" * 50)
print("Batch images shape :", images.shape)
print("Batch labels shape :", labels.shape)
print()

print("Image dtype :", images.dtype)
print("Label dtype :", labels.dtype)
print()

print("Unique labels :", torch.unique(labels).tolist())
print()

print("Image value range")
print("Min :", images.min().item())
print("Max :", images.max().item())
print()

print("First 10 labels:")
print(labels[:10].tolist())
print("=" * 50)


Batch images shape : torch.Size([32, 3, 224, 224])
Batch labels shape : torch.Size([32])

Image dtype : torch.float32
Label dtype : torch.int64

Unique labels : [0, 1]

Image value range
Min : -2.1179039478302
Max : 2.640000104904175

First 10 labels:
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0]
